# Stage FLARE site-filter resources

Builds Part 4 / Part 7 filter files for `FlareByPopulation` /
`flare/configs/lai_exp.tsv`:

| Resource | Column | Product |
|---|---|---|
| Context mask | `exclude_regions` | Union BED: UCSC `rmsk` ∪ `simpleRepeat` ∪ `genomicSuperDups` |
| Concordance sites (optional / deferred) | `include_sites` | HiFi ↔ srWGS genotype-concordant biallelic SNVs |
| Call-QC sites (Part 7.2) | `include_sites` | Biallelic SNVs with low frac GQ/DP/RNC failures |

Ladder rows: context mask is live; `chr22_filter_call_qc` /
`chr20_filter_call_qc` need Part C below. Concordance Part B remains deferred.

**Indel flanks are not in the context mask.** Ladder rows already set
`indel_flank_bp=5`; the WDL merges those at runtime via
`flare_build_indel_flanks.py`.

## Outputs (default prefix)

`$WORKSPACE_BUCKET/refs/flare/`

- `flare_context_mask.rmsk_sr_sd.bed.gz` (+ `.tbi` when tabix is available)
- `flare_context_mask.manifest.json`
- `hifi_srwgs_concordant.<region>.bed.gz` (+ stats `.tsv.gz`) — Part B only
- `call_qc.<region>.bed.gz` (+ stats `.tsv.gz`) — Part C

## Prerequisites

- Terra notebook VM with `gsutil`, network to UCSC (unless beds already staged)
- Prefer reusing beds from `sv_01_stage_repeat_tracks.ipynb` under
  `$WORKSPACE_BUCKET/refs/grch38/` when present
- Part B: `bcftools`, HiFi + srWGS VCFs; upload `flare_build_concordance_sites.py`
- Part C: `bcftools`, DeepVariant/GLnexus joint VCF with FORMAT GQ/DP/RNC;
  upload `flare_build_call_qc_sites.py`

Set `FLARE_FILTER_DRY_RUN=true` to build locally without uploading.

In [ ]:
from __future__ import annotations

from pathlib import Path
import gzip
import json
import os
import shutil
import subprocess
import sys
import urllib.request
from datetime import datetime, timezone

for _d in (Path.cwd() / "scripts", Path.cwd().parent / "scripts", Path.cwd().parent.parent / "scripts"):
    if (_d / "terra_notebook.py").is_file():
        sys.path.insert(0, str(_d.resolve()))
        break
else:
    _bucket = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
    if not _bucket:
        raise FileNotFoundError(
            "scripts/ not found locally and WORKSPACE_BUCKET is unset. "
            "Upload scripts/ to gs://WORKSPACE/scripts/."
        )

from terra_notebook import init_notebook, env_flag

SCRIPTS = init_notebook(
    "flare_build_concordance_sites.py",
    "flare_build_call_qc_sites.py",
)
print("SCRIPTS:", SCRIPTS)

In [ ]:
UCSC_DB = "https://hgdownload.soe.ucsc.edu/goldenPath/hg38/database"
MIN_FREE_GB = 5

TRACKS = [
    {
        "name": "rmsk",
        "url": f"{UCSC_DB}/rmsk.txt.gz",
        "chrom_col": 5,
        "start_col": 6,
        "end_col": 7,
        "staged_name": "rmsk.bed.gz",
    },
    {
        "name": "simpleRepeat",
        "url": f"{UCSC_DB}/simpleRepeat.txt.gz",
        "chrom_col": 1,
        "start_col": 2,
        "end_col": 3,
        "staged_name": "simpleRepeat.bed.gz",
    },
    {
        "name": "genomicSuperDups",
        "url": f"{UCSC_DB}/genomicSuperDups.txt.gz",
        "chrom_col": 1,
        "start_col": 2,
        "end_col": 3,
        "staged_name": "genomicSuperDups.bed.gz",
    },
]

WORK = Path(os.environ.get("FLARE_FILTER_WORK", Path.cwd() / "flare_site_filter_work")).resolve()
RAW = WORK / "raw"
BEDS = WORK / "beds"
OUT = WORK / "out"

WORKSPACE_BUCKET = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
STAGED_REPEAT_PREFIX = os.environ.get(
    "REPEAT_TRACK_GCS_PREFIX",
    f"{WORKSPACE_BUCKET}/refs/grch38" if WORKSPACE_BUCKET else "",
).rstrip("/")
GCS_PREFIX = os.environ.get(
    "FLARE_FILTER_GCS_PREFIX",
    f"{WORKSPACE_BUCKET}/refs/flare" if WORKSPACE_BUCKET else "",
).rstrip("/")
DRY_RUN = env_flag("FLARE_FILTER_DRY_RUN", default=False)
BUILD_CONTEXT = env_flag("FLARE_BUILD_CONTEXT_MASK", default=True)
BUILD_CONCORDANCE = env_flag("FLARE_BUILD_CONCORDANCE", default=False)
BUILD_CALL_QC = env_flag("FLARE_BUILD_CALL_QC", default=False)

# Part B — required when FLARE_BUILD_CONCORDANCE=true
HIFI_VCF = os.environ.get("FLARE_CONCORDANCE_HIFI_VCF", "").strip()
SR_VCF = os.environ.get("FLARE_CONCORDANCE_SR_VCF", "").strip()
CONCORDANCE_REGION = os.environ.get("FLARE_CONCORDANCE_REGION", "chr20").strip()
CONCORDANCE_SAMPLES = os.environ.get("FLARE_CONCORDANCE_SAMPLES", "").strip()
MIN_CONCORDANCE = float(os.environ.get("FLARE_MIN_CONCORDANCE", "1.0"))
MIN_CALLED = int(os.environ.get("FLARE_MIN_CALLED", "10"))

# Part C — required when FLARE_BUILD_CALL_QC=true (DeepVariant/GLnexus joint VCF)
CALL_QC_VCF = os.environ.get("FLARE_CALL_QC_VCF", "").strip()
CALL_QC_REGION = os.environ.get("FLARE_CALL_QC_REGION", "chr22:26897597-36897597").strip()
CALL_QC_SAMPLES = os.environ.get("FLARE_CALL_QC_SAMPLES", "").strip()
CALL_QC_MIN_GQ = int(os.environ.get("FLARE_CALL_QC_MIN_GQ", "20"))
CALL_QC_MIN_DP = int(os.environ.get("FLARE_CALL_QC_MIN_DP", "10"))
CALL_QC_MAX_FRAC_FAIL = float(os.environ.get("FLARE_CALL_QC_MAX_FRAC_FAIL", "0.05"))

CONTEXT_OUT_NAME = "flare_context_mask.rmsk_sr_sd.bed.gz"

print("WORK:", WORK)
print("STAGED_REPEAT_PREFIX:", STAGED_REPEAT_PREFIX or "(none)")
print("GCS_PREFIX:", GCS_PREFIX or "(none)")
print("DRY_RUN:", DRY_RUN)
print(
    "BUILD_CONTEXT:", BUILD_CONTEXT,
    "BUILD_CONCORDANCE:", BUILD_CONCORDANCE,
    "BUILD_CALL_QC:", BUILD_CALL_QC,
)
if BUILD_CONCORDANCE:
    print("HIFI_VCF:", HIFI_VCF or "(required)")
    print("SR_VCF:", SR_VCF or "(required)")
    print("REGION:", CONCORDANCE_REGION)
if BUILD_CALL_QC:
    print("CALL_QC_VCF:", CALL_QC_VCF or "(required)")
    print("CALL_QC_REGION:", CALL_QC_REGION)
    print("min_gq/min_dp/max_frac_fail:", CALL_QC_MIN_GQ, CALL_QC_MIN_DP, CALL_QC_MAX_FRAC_FAIL)

In [ ]:
def run(cmd: list[str], *, check: bool = True) -> subprocess.CompletedProcess[str]:
    print("+", " ".join(map(str, cmd)))
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.stdout:
        print(proc.stdout, end="" if proc.stdout.endswith("\n") else "\n")
    if proc.stderr:
        print(proc.stderr, end="" if proc.stderr.endswith("\n") else "\n")
    if check and proc.returncode != 0:
        raise subprocess.CalledProcessError(
            proc.returncode, cmd, output=proc.stdout, stderr=proc.stderr
        )
    return proc


def gcs_exists(uri: str) -> bool:
    proc = subprocess.run(["gsutil", "-q", "stat", uri], capture_output=True, text=True)
    return proc.returncode == 0


def download(url: str, dest: Path) -> Path:
    if dest.exists() and dest.stat().st_size > 0:
        print(f"Reusing {dest}")
        return dest
    tmp = dest.with_suffix(dest.suffix + ".partial")
    print(f"Downloading {url}")
    urllib.request.urlretrieve(url, tmp)
    tmp.replace(dest)
    print(f"Wrote {dest} ({dest.stat().st_size / 1e6:.1f} MB)")
    return dest


def dump_to_bed(src: Path, dest: Path, chrom_col: int, start_col: int, end_col: int) -> int:
    n = 0
    with gzip.open(src, "rt") as fh, dest.open("w") as out:
        for line in fh:
            if not line.strip() or line.startswith("#"):
                continue
            parts = line.rstrip("\n").split("\t")
            needed = max(chrom_col, start_col, end_col)
            if len(parts) <= needed:
                continue
            chrom = parts[chrom_col].strip()
            if not chrom:
                continue
            start = int(parts[start_col])
            end = int(parts[end_col])
            if end < start:
                start, end = end, start
            if end == start:
                end = start + 1
            out.write(f"{chrom}\t{start}\t{end}\n")
            n += 1
    return n


def compress_bed(sorted_bed: Path, out_gz: Path) -> None:
    if out_gz.exists():
        out_gz.unlink()
    tbi = Path(str(out_gz) + ".tbi")
    if tbi.exists():
        tbi.unlink()
    if shutil.which("bgzip"):
        with out_gz.open("wb") as fh:
            subprocess.run(["bgzip", "-c", str(sorted_bed)], stdout=fh, check=True)
        if shutil.which("tabix"):
            run(["tabix", "-p", "bed", str(out_gz)])
    else:
        with sorted_bed.open("rb") as src, gzip.open(out_gz, "wb") as dest:
            shutil.copyfileobj(src, dest)


def merge_beds(paths: list[Path], out_bed: Path) -> int:
    cat = BEDS / "_cat.bed"
    with cat.open("w") as out:
        for path in paths:
            opener = gzip.open if str(path).endswith(".gz") else open
            with opener(path, "rt") as fh:  # type: ignore[arg-type]
                for line in fh:
                    if line.strip() and not line.startswith("#"):
                        out.write(line if line.endswith("\n") else line + "\n")
    sorted_bed = BEDS / "_sorted.bed"
    run(["sort", "-k1,1", "-k2,2n", "-o", str(sorted_bed), str(cat)])
    if shutil.which("bedtools"):
        print("+ bedtools merge -i", sorted_bed)
        proc = subprocess.run(
            ["bedtools", "merge", "-i", str(sorted_bed)],
            capture_output=True,
            text=True,
            check=True,
        )
        out_bed.write_text(proc.stdout)
    else:
        merged: list[tuple[str, int, int]] = []
        with sorted_bed.open() as fh:
            for line in fh:
                chrom, start_s, end_s = line.rstrip("\n").split("\t")[:3]
                start, end = int(start_s), int(end_s)
                if not merged or merged[-1][0] != chrom or start > merged[-1][2]:
                    merged.append((chrom, start, end))
                else:
                    prev = merged[-1]
                    merged[-1] = (prev[0], prev[1], max(prev[2], end))
        with out_bed.open("w") as out:
            for chrom, start, end in merged:
                out.write(f"{chrom}\t{start}\t{end}\n")
    cat.unlink(missing_ok=True)
    sorted_bed.unlink(missing_ok=True)
    return sum(1 for _ in out_bed.open() if _.strip())


for d in (WORK, RAW, BEDS, OUT):
    d.mkdir(parents=True, exist_ok=True)

available = shutil.disk_usage(WORK).free / (1024 ** 3)
print(f"Free disk at {WORK}: {available:.1f} GB")
if available < MIN_FREE_GB and BUILD_CONTEXT:
    print(
        f"Warning: <{MIN_FREE_GB} GB free; ok if reusing staged beds, "
        "otherwise download may fail."
    )
print("bgzip:", shutil.which("bgzip") is not None, "tabix:", shutil.which("tabix") is not None, "bedtools:", shutil.which("bedtools") is not None, "bcftools:", shutil.which("bcftools") is not None)

## Part A — context mask (`exclude_regions`)

Reuse `$WORKSPACE_BUCKET/refs/grch38/{rmsk,simpleRepeat,genomicSuperDups}.bed.gz`
when present; otherwise download UCSC dumps and convert (same recipe as
`sv_01_stage_repeat_tracks.ipynb`). Merge to a single sorted BED and upload.

In [ ]:
context_uri = None
component_beds: list[Path] = []
manifest = {
    "genome_build": "GRCh38",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "definition": "union(rmsk, simpleRepeat, genomicSuperDups)",
    "notes": (
        "FLARE exclude_regions context mask. Indel flanks are added at runtime "
        "by PrepExcludeRegions when indel_flank_bp > 0."
    ),
    "components": [],
}

if BUILD_CONTEXT:
    for track in TRACKS:
        local_gz = BEDS / track["staged_name"]
        staged_uri = f"{STAGED_REPEAT_PREFIX}/{track['staged_name']}" if STAGED_REPEAT_PREFIX else ""
        sourced = None
        if staged_uri and gcs_exists(staged_uri):
            print(f"Fetching staged {staged_uri}")
            run(["gsutil", "-m", "cp", staged_uri, str(local_gz)])
            tbi_uri = staged_uri + ".tbi"
            if gcs_exists(tbi_uri):
                run(["gsutil", "-m", "cp", tbi_uri, str(local_gz) + ".tbi"])
            sourced = staged_uri
        else:
            raw = download(track["url"], RAW / f"{track['name']}.txt.gz")
            unsorted = BEDS / f"{track['name']}.unsorted.bed"
            sorted_bed = BEDS / f"{track['name']}.bed"
            n = dump_to_bed(raw, unsorted, track["chrom_col"], track["start_col"], track["end_col"])
            print(f"{track['name']}: {n:,} intervals")
            if n == 0:
                raise SystemExit(f"No intervals from {raw}")
            run(["sort", "-k1,1", "-k2,2n", "-o", str(sorted_bed), str(unsorted)])
            compress_bed(sorted_bed, local_gz)
            unsorted.unlink(missing_ok=True)
            sorted_bed.unlink(missing_ok=True)
            sourced = track["url"]
        component_beds.append(local_gz)
        manifest["components"].append(
            {"name": track["name"], "source": sourced, "file": track["staged_name"]}
        )

    merged_bed = OUT / "flare_context_mask.rmsk_sr_sd.bed"
    n_merged = merge_beds(component_beds, merged_bed)
    print(f"Merged intervals: {n_merged:,}")
    context_gz = OUT / CONTEXT_OUT_NAME
    compress_bed(merged_bed, context_gz)
    merged_bed.unlink(missing_ok=True)
    manifest["n_merged_intervals"] = n_merged
    manifest["file"] = CONTEXT_OUT_NAME
    manifest_path = OUT / "flare_context_mask.manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
    print("Wrote", context_gz, "and", manifest_path)

    if DRY_RUN:
        print("DRY_RUN: skip upload")
        context_uri = str(context_gz)
    else:
        if not GCS_PREFIX:
            raise SystemExit("Set WORKSPACE_BUCKET or FLARE_FILTER_GCS_PREFIX to upload")
        context_uri = f"{GCS_PREFIX}/{CONTEXT_OUT_NAME}"
        run(["gsutil", "-m", "cp", str(context_gz), context_uri])
        tbi = Path(str(context_gz) + ".tbi")
        if tbi.exists():
            run(["gsutil", "-m", "cp", str(tbi), context_uri + ".tbi"])
        run(["gsutil", "-m", "cp", str(manifest_path), f"{GCS_PREFIX}/flare_context_mask.manifest.json"])
        print("Uploaded", context_uri)
else:
    print("Skipping context mask (FLARE_BUILD_CONTEXT_MASK=false)")

## Part B — concordance sites (`include_sites`)

Off by default. Enable with `FLARE_BUILD_CONCORDANCE=true` and set:

| Env | Meaning |
|---|---|
| `FLARE_CONCORDANCE_HIFI_VCF` | Phased HiFi / long-read target (e.g. `aou_lr_phase2_v1.chr20.vcf.gz`) |
| `FLARE_CONCORDANCE_SR_VCF` | Short-read WGS VCF for **overlapping** `research_id`s |
| `FLARE_CONCORDANCE_REGION` | Default `chr20` |
| `FLARE_CONCORDANCE_SAMPLES` | Optional keep-list path/URI |
| `FLARE_MIN_CONCORDANCE` | Default `1.0` (perfect GT agreement among callable pairs) |
| `FLARE_MIN_CALLED` | Default `10` shared samples with non-missing GT on both sides |

Uses `scripts/flare_build_concordance_sites.py`. This is **not** the gnomAD LAI
panel — it must be srWGS genotypes for the same people.

In [ ]:
concordance_uri = None

if BUILD_CONCORDANCE:
    if not HIFI_VCF or not SR_VCF:
        raise SystemExit(
            "FLARE_BUILD_CONCORDANCE=true requires FLARE_CONCORDANCE_HIFI_VCF and "
            "FLARE_CONCORDANCE_SR_VCF"
        )
    if shutil.which("bcftools") is None:
        raise SystemExit("bcftools is required for concordance site building")

    def localize(uri_or_path: str, dest_dir: Path) -> Path:
        if uri_or_path.startswith("gs://"):
            dest = dest_dir / Path(uri_or_path).name
            if not dest.exists():
                run(["gsutil", "-m", "cp", uri_or_path, str(dest)])
            # sibling index if present
            for suf in (".tbi", ".csi"):
                idx = uri_or_path + suf
                if gcs_exists(idx):
                    run(["gsutil", "-m", "cp", idx, str(dest) + suf])
            return dest
        path = Path(uri_or_path)
        if not path.exists():
            raise FileNotFoundError(path)
        return path

    hifi_local = localize(HIFI_VCF, RAW)
    sr_local = localize(SR_VCF, RAW)
    sample_arg: list[str] = []
    if CONCORDANCE_SAMPLES:
        samples_path = localize(CONCORDANCE_SAMPLES, RAW)
        sample_arg = ["--samples", str(samples_path)]

    region_tag = CONCORDANCE_REGION.replace(":", "_").replace("-", "_") or "genome"
    out_prefix = OUT / f"hifi_srwgs_concordant.{region_tag}"
    cmd = [
        sys.executable,
        str(SCRIPTS / "flare_build_concordance_sites.py"),
        "--hifi-vcf",
        str(hifi_local),
        "--sr-vcf",
        str(sr_local),
        "--region",
        CONCORDANCE_REGION,
        "--min-concordance",
        str(MIN_CONCORDANCE),
        "--min-called",
        str(MIN_CALLED),
        "--out-prefix",
        str(out_prefix),
        *sample_arg,
    ]
    run(cmd)
    bed_gz = Path(str(out_prefix) + ".bed.gz")
    tsv_gz = Path(str(out_prefix) + ".tsv.gz")
    if DRY_RUN:
        concordance_uri = str(bed_gz)
        print("DRY_RUN: skip upload")
    else:
        if not GCS_PREFIX:
            raise SystemExit("Set WORKSPACE_BUCKET or FLARE_FILTER_GCS_PREFIX to upload")
        concordance_uri = f"{GCS_PREFIX}/{bed_gz.name}"
        run(["gsutil", "-m", "cp", str(bed_gz), concordance_uri])
        tbi = Path(str(bed_gz) + ".tbi")
        if tbi.exists():
            run(["gsutil", "-m", "cp", str(tbi), concordance_uri + ".tbi"])
        run(["gsutil", "-m", "cp", str(tsv_gz), f"{GCS_PREFIX}/{tsv_gz.name}"])
        print("Uploaded", concordance_uri)
else:
    print(
        "Skipping concordance (set FLARE_BUILD_CONCORDANCE=true and provide HiFi + srWGS VCFs)."
    )

## Part C — call-QC sites (`include_sites`, Part 7.2)

Off by default. Enable with `FLARE_BUILD_CALL_QC=true` and set
`FLARE_CALL_QC_VCF` to a DeepVariant/GLnexus joint VCF that has FORMAT
`GQ`/`DP`/`RNC`. Default region is the chr22 10 Mb method window.

Uses `scripts/flare_build_call_qc_sites.py`. Keep sites where the fraction of
QC-able samples failing GQ/DP/RNC is ≤ `FLARE_CALL_QC_MAX_FRAC_FAIL` (default
0.05). Paste the resulting URI into `chr22_filter_call_qc` /
`chr20_filter_call_qc` `include_sites`.

Do **not** confuse with Part B concordance beds — both use `include_sites`,
but they answer different questions.

In [ ]:
call_qc_uri = None

if BUILD_CALL_QC:
    if not CALL_QC_VCF:
        raise SystemExit("Set FLARE_CALL_QC_VCF to a GLnexus/DeepVariant joint VCF")
    if not shutil.which("bcftools"):
        raise SystemExit("bcftools is required for call-QC site building")
    OUT.mkdir(parents=True, exist_ok=True)
    region_tag = CALL_QC_REGION.replace(":", "_").replace("-", "_") or "genome"
    out_prefix = OUT / f"call_qc.{region_tag}"
    cmd = [
        sys.executable,
        str(SCRIPTS / "flare_build_call_qc_sites.py"),
        "--vcf",
        CALL_QC_VCF,
        "--region",
        CALL_QC_REGION,
        "--min-gq",
        str(CALL_QC_MIN_GQ),
        "--min-dp",
        str(CALL_QC_MIN_DP),
        "--max-frac-fail",
        str(CALL_QC_MAX_FRAC_FAIL),
        "--out-prefix",
        str(out_prefix),
    ]
    if CALL_QC_SAMPLES:
        keep = OUT / "call_qc.keep.txt"
        keep.write_text(Path(CALL_QC_SAMPLES).read_text() if Path(CALL_QC_SAMPLES).is_file() else CALL_QC_SAMPLES.replace(",", "\n") + "\n")
        cmd.extend(["--keep", str(keep)])
    run(cmd)
    bed_gz = Path(str(out_prefix) + ".bed.gz")
    if not bed_gz.is_file():
        # script may leave .bed then bgzip
        bed = Path(str(out_prefix) + ".bed")
        if bed.is_file():
            bed_gz = bed
    if DRY_RUN or not GCS_PREFIX:
        call_qc_uri = str(bed_gz)
        print("DRY_RUN / no GCS_PREFIX; local", call_qc_uri)
    else:
        call_qc_uri = f"{GCS_PREFIX}/{bed_gz.name}"
        run(["gsutil", "-m", "cp", str(bed_gz), call_qc_uri])
        tbi = Path(str(bed_gz) + ".tbi")
        if tbi.is_file():
            run(["gsutil", "-m", "cp", str(tbi), call_qc_uri + ".tbi"])
        stats = Path(str(out_prefix) + ".tsv.gz")
        if stats.is_file():
            run(["gsutil", "-m", "cp", str(stats), f"{GCS_PREFIX}/{stats.name}"])
        print("Uploaded", call_qc_uri)
else:
    print("Skipping call-QC (set FLARE_BUILD_CALL_QC=true and FLARE_CALL_QC_VCF).")

## Paste into `lai_exp.tsv`

After a successful upload:

- set `exclude_regions` on context-mask ladder rows
- set `include_sites` on `chr22_filter_call_qc` / `chr20_filter_call_qc` from Part C
- (optional) concordance Part B URI when that ladder step is revived

In [ ]:
print("=== lai_exp.tsv column values ===")
if context_uri:
    print(f"exclude_regions:\n  {context_uri}")
else:
    print("exclude_regions: (not built this run)")
if concordance_uri:
    print(f"include_sites (concordance / deferred):\n  {concordance_uri}")
if call_qc_uri:
    print(f"include_sites (call-QC → chr22_filter_call_qc / chr20_filter_call_qc):\n  {call_qc_uri}")
if not concordance_uri and not call_qc_uri:
    print("include_sites: skipped (enable Part B and/or Part C)")

if GCS_PREFIX and not DRY_RUN:
    print("\nBucket listing:")
    run(["gsutil", "ls", "-lh", f"{GCS_PREFIX}/"], check=False)

## Optional cleanup

In [ ]:
# shutil.rmtree(WORK)
# print("Removed", WORK)